In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('../data/fake_job_postings.csv')


In [3]:
text_columns = ['title', 'description', 'requirements', 'company_profile', 'benefits']

df[text_columns] = df[text_columns].fillna('')

In [4]:
df[text_columns].isnull().sum()

title              0
description        0
requirements       0
company_profile    0
benefits           0
dtype: int64

In [5]:
df['combined_text'] = (df['title'] + ' ' + 
                       df['description'] + ' ' + 
                       df['requirements'] + ' ' + 
                       df['company_profile'])

In [6]:
print(df['combined_text'][0])

Marketing Intern Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its New York City headquarters.Reproducing and/or repackaging existing Food52 content for a number of partner sites, such as Huffington Post, Yahoo, Buzzfeed, and more in their various content management systemsResearching blogs and websites for the Provisions by Food52 Affiliate ProgramAssisting in day-to-day affiliate program support, such as screening affiliates and assisting in any affiliate inquiriesSupporting with PR &amp; Events when neededHelping with office administrative work, such as filing, mailing, and preparing for meetingsWorking with developers to document bugs and suggest improvements to the siteSupporting the marketing and executive staff Experience with content management systems a major plus (any blogging counts!

In [7]:
df = df.drop(columns=['job_id', 'salary_range', 'department', 
                       'location', 'function', 'industry',
                       'benefits', 'title', 'description', 
                       'requirements', 'company_profile'])

In [8]:
df.columns

Index(['telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'fraudulent',
       'combined_text'],
      dtype='str')

In [9]:
cat_columns = ['employment_type', 'required_experience','required_education']
df[cat_columns] = df[cat_columns].fillna('Unknown')

In [10]:
df.isnull().sum()

telecommuting          0
has_company_logo       0
has_questions          0
employment_type        0
required_experience    0
required_education     0
fraudulent             0
combined_text          0
dtype: int64

In [11]:
df = pd.get_dummies(df,columns = ['employment_type','required_experience','required_education'])

In [12]:
print(df.shape)
print(df.columns.tolist())

(17880, 33)
['telecommuting', 'has_company_logo', 'has_questions', 'fraudulent', 'combined_text', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'employment_type_Unknown', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_experience_Unknown', 'required_education_Associate Degree', "required_education_Bachelor's Degree", 'required_education_Certification', 'required_education_Doctorate', 'required_education_High School or equivalent', "required_education_Master's Degree", 'required_education_Professional', 'required_education_Some College Coursework Completed', 'required_education_Some High School Coursework', 'required_education_Unknown', 'required_education_Unspecified', 'required_education_V

In [13]:
df = df.astype({col: int for col in df.select_dtypes('bool').columns})
print(df.dtypes.value_counts())

int64    32
str       1
Name: count, dtype: int64


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 33 columns):
 #   Column                                                Non-Null Count  Dtype
---  ------                                                --------------  -----
 0   telecommuting                                         17880 non-null  int64
 1   has_company_logo                                      17880 non-null  int64
 2   has_questions                                         17880 non-null  int64
 3   fraudulent                                            17880 non-null  int64
 4   combined_text                                         17880 non-null  str  
 5   employment_type_Contract                              17880 non-null  int64
 6   employment_type_Full-time                             17880 non-null  int64
 7   employment_type_Other                                 17880 non-null  int64
 8   employment_type_Part-time                             17880 non-null  int64
 9   employ

In [15]:
X_text = df['combined_text']
X_numeric = df.drop(columns=['fraudulent', 'combined_text'])
y = df['fraudulent']

print(X_text.shape)
print(X_numeric.shape)
print(y.shape)

(17880,)
(17880, 31)
(17880,)


In [16]:
from sklearn.model_selection import train_test_split

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_text_train.shape)
print(X_text_test.shape)
print(y_train.value_counts())

(14304,)
(3576,)
fraudulent
0    13611
1      693
Name: count, dtype: int64


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range = (1,2))

X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf = tfidf.transform(X_text_test)

print(X_text_train_tfidf.shape)
print(X_text_test_tfidf.shape)

(14304, 5000)
(3576, 5000)


In [18]:
from scipy.sparse import hstack
import numpy as np

X_train_final = hstack([X_text_train_tfidf, X_num_train])
X_test_final = hstack([X_text_test_tfidf, X_num_test])

print(X_train_final.shape)
print(X_test_final.shape)

(14304, 5031)
(3576, 5031)


In [20]:
import pickle

with open('../backend/app/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('TF-IDF saved!')

TF-IDF saved!
